# SQL DB

Converting CSV DB of patients (Before Cleaning for training) into a SQL DB, which we can query with SQLLite

In [6]:
import pandas as pd, os
import sqlite3

# 1. Load CSV file
filename = "patients_test_structured.csv"
#path = filename if os.path.exists(filename) else os.path.join("..", filename) 
filepath_in = filename
df = pd.read_csv(filepath_in)

# 2. Connect to (or create) a SQLite database + CONFIG
filepath_out = "patients_test_structured.db"
conn = sqlite3.connect(filepath_out)

pd.set_option('display.max_rows', None)

# 3. Write the data to a SQL table
df.to_sql("patients", conn, if_exists="replace", index=False)

# 4. Commit and close
conn.commit()
conn.close()

In [14]:
#Now, we can launch queries with sqlite3
filepath_in = filepath_out
conn = sqlite3.connect(filepath_in)

query_all = """
SELECT *
FROM patients
"""
query_eliminated = """
SELECT *
FROM patients
WHERE patient_id NOT IN (
    SELECT patient_id
    FROM patients
    WHERE Age > -1 AND Gender IN ('Male', 'Female') AND Hypertension IN (0, 1) AND
    [Heart Disease] IN (0, 1) AND [Smoking History] IN ('Non-smoker', 'Current', 'Past') AND
    BMI > 0 AND HbA1c > 0 AND [Random Glucose] > 0
    )
"""
query_remaining = """
SELECT COUNT(*)
FROM patients
WHERE patient_id IN (
    SELECT patient_id
    FROM patients
    WHERE Age > -1 AND Gender IN ('Male', 'Female') AND Hypertension IN (0, 1) AND
    [Heart Disease] IN (0, 1) AND [Smoking History] IN ('Non-smoker', 'Current', 'Past') AND
    BMI > 0 AND HbA1c > 0 AND [Random Glucose] > 0
    ) 
"""

query_eliminated_diabetic = """
SELECT COUNT(*)
FROM patients
WHERE patient_id NOT IN (
    SELECT patient_id
    FROM patients
    WHERE Age > -1 AND Gender IN ('Male', 'Female') AND Hypertension IN (0, 1) AND
    [Heart Disease] IN (0, 1) AND [Smoking History] IN ('Non-smoker', 'Current', 'Past') AND
    BMI > 0 AND HbA1c > 0 AND [Random Glucose] > 0
    ) AND has_diabetes = 1
"""
query_eliminated_non_diabetic = """
SELECT COUNT(*)
FROM patients
WHERE patient_id NOT IN (
    SELECT patient_id
    FROM patients
    WHERE Age > -1 AND Gender IN ('Male', 'Female') AND Hypertension IN (0, 1) AND
    [Heart Disease] IN (0, 1) AND [Smoking History] IN ('Non-smoker', 'Current', 'Past') AND
    BMI > 0 AND HbA1c > 0 AND [Random Glucose] > 0
    ) AND has_diabetes = 0
"""
query_remaining_diabetic = """
SELECT COUNT(*)
FROM patients
WHERE patient_id IN (
    SELECT patient_id
    FROM patients
    WHERE Age > -1 AND Gender IN ('Male', 'Female') AND Hypertension IN (0, 1) AND
    [Heart Disease] IN (0, 1) AND [Smoking History] IN ('Non-smoker', 'Current', 'Past') AND
    BMI > 0 AND HbA1c > 0 AND [Random Glucose] > 0
    ) AND has_diabetes = 1
"""
query_remaining_non_diabetic = """
SELECT COUNT(*)
FROM patients
WHERE patient_id IN (
    SELECT patient_id
    FROM patients
    WHERE Age > -1 AND Gender IN ('Male', 'Female') AND Hypertension IN (0, 1) AND
    [Heart Disease] IN (0, 1) AND [Smoking History] IN ('Non-smoker', 'Current', 'Past') AND
    BMI > 0 AND HbA1c > 0 AND [Random Glucose] > 0
    ) AND has_diabetes = 0
"""

query_eliminated_floats = """
SELECT COUNT(*)
FROM patients
WHERE patient_id NOT IN (
    SELECT patient_id
    FROM patients
    WHERE HbA1c > 0 AND [Random Glucose] > 0
    )
"""

query = """
SELECT *
FROM patients
WHERE patient_id NOT IN (
    SELECT patient_id
    FROM patients
    WHERE Age > -1 AND Gender IN ('Male', 'Female') AND Hypertension IN (0, 1) AND
    [Heart Disease] IN (0, 1) AND [Smoking History] IN ('Non-smoker', 'Current', 'Past') 
    AND BMI > 0 
    )
"""
# HbA1c > 0
# [Random Glucose] > 0
query = query
pd.read_sql(query, conn)

,patient_id,Age,Gender,Hypertension,Heart Disease,Smoking History,BMI,HbA1c,Random Glucose
0,68425,70,Female,NaN,1.0,Past,32.16,5.3,160.0
1,76598,66,Female,1.0,NaN,Non-smoker,41.00,6.6,220.0
2,9183,65,Male,NaN,1.0,Past,27.98,5.8,240.0
3,29689,6,Male,0.0,0.0,None,16.35,4.0,158.0
4,72455,36,Female,NaN,NaN,Past,31.91,5.3,85.0
5,8382,31,Female,1.0,NaN,Past,27.30,7.5,160.0
6,87076,77,Female,NaN,1.0,Current,23.24,5.8,160.0
7,88977,45,Female,NaN,1.0,Current,20.31,6.2,160.0
8,78336,62,Male,1.0,NaN,Past,37.10,7.0,126.0
9,11825,73,Male,1.0,NaN,Non-smoker,37.00,6.8,90.0
